In [ ]:
# pip install pandas scikit-learn matplotlib seaborn

import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns


In [ ]:

# --- Step 1: Download dataset directly (UCI, no auth needed) ---
url = "https://archive.ics.uci.edu/ml/machine-learning-databases/wine-quality/winequality-red.csv"
df = pd.read_csv(url, sep=';')

# --- Step 2: Turn quality score (3-8) into 3 classes ---
def quality_to_class(q):
    if q <= 4:
        return 'low'
    elif q <= 6:
        return 'medium'
    else:
        return 'high'

df['quality_class'] = df['quality'].apply(quality_to_class)

X = df.drop(columns=['quality', 'quality_class'])
y = df['quality_class']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# --- Step 3: Train multinomial logistic regression ---
model = LogisticRegression(multi_class='multinomial', solver='lbfgs',
                            max_iter=1000, class_weight='balanced')
model.fit(X_train_scaled, y_train)

# --- Step 4: Evaluate ---
y_pred = model.predict(X_test_scaled)
print(classification_report(y_test, y_pred))

cm = confusion_matrix(y_test, y_pred, labels=model.classes_)
sns.heatmap(cm, annot=True, fmt='d', xticklabels=model.classes_, yticklabels=model.classes_)
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.title("Confusion Matrix - Wine Quality")
plt.show()

# --- Step 5: Compare with One-vs-Rest ---
model_ovr = LogisticRegression(multi_class='ovr', solver='liblinear',
                                max_iter=1000, class_weight='balanced')
model_ovr.fit(X_train_scaled, y_train)
y_pred_ovr = model_ovr.predict(X_test_scaled)
print("--- One-vs-Rest ---")
print(classification_report(y_test, y_pred_ovr))